### `1.) DATA LOADING AND DATA CLEANING`

IMPORT LIBRARIES        

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

FILTER OUT 21 COLUMNS from 145 COLUMNS that we are going to use.

At this stage we load only a subset of columns relevant for credit risk modelling instead of all ~145 available variables.  
This helps reduce memory usage and focuses the analysis on borrower characteristics and loan attributes available at origination.

In [9]:
cols = [
"loan_amnt",
"term",
"int_rate",
"installment",
"grade",
"sub_grade",
"emp_length",
"home_ownership",
"annual_inc",
"verification_status",
"issue_d",
"loan_status",
"purpose",
"dti",
"fico_range_low",
"fico_range_high",
"open_acc",
"revol_bal",
"revol_util",
"total_acc",
"recoveries"
]

### Initial Dataset Load

The LendingClub dataset contains detailed records of loans issued on the platform between 2007 and 2018.

The dataset initially contains over 2 million loan records.

In [10]:
df = pd.read_csv(
"accepted_2007_to_2018Q4.csv",
usecols=cols,
low_memory=False
)

df.shape

(2260701, 21)

Total loans	2,260,701

Variables used	21

LOAN STATUS DISTRIBUTION

In [11]:
df["loan_status"].value_counts()

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

The `loan_status` variable describes the final or current state of each loan.

For credit risk modelling we must use loans whose **final outcome is known**.  
Loans that are still active (e.g. "Current" or "Late") have not yet reached their final state and therefore cannot be used to train a default prediction model.

The next step filters the dataset to include **only completed loans**.


KEEP ONLY COMPLETED LOANS

These filters will be for loans whose final outcome is known.
We will train only on this data.

- **Fully Paid** → borrower successfully repaid the loan
- **Charged Off** → borrower defaulted and the loan was written off

This step ensures the model learns from **resolved credit events**, which is essential for estimating probability of default.

In [12]:
df = df[df["loan_status"].isin(["Fully Paid", "Charged Off"])]

In [13]:
df.shape

(1345310, 21)

Category	   |       Loans

Fully Paid	   |       ~1.08M

Charged Off	   |      ~268k


1.34M loans
20 predictor variables
1 target variable

CREATE TARGET VARIABLE : LOAN DEFAULT STATUS

Credit risk models require a binary target variable indicating whether a borrower defaulted.

We create a new variable called `default`:

- **1 = Default (Charged Off)**
- **0 = Non-default (Fully Paid)**

In [14]:
df["default"] = np.where(df["loan_status"] == "Charged Off", 1, 0)

DROP LOAN STATUS

After creating the `default` variable, the original `loan_status` column is removed because its information has already been encoded into the binary target.

In [15]:
df.drop("loan_status", axis=1, inplace=True)

CHECK LOAN DEFAULT RATE

Before building predictive models it is important to examine the distribution of the target variable.

This shows the proportion of loans that resulted in default versus successful repayment.

Credit datasets typically exhibit **class imbalance**, where non-default loans significantly outnumber defaulted loans. 

Understanding this imbalance helps guide model evaluation and performance metrics later in the analysis.

In [16]:
df["default"].value_counts(normalize=True)

default
0    0.800374
1    0.199626
Name: proportion, dtype: float64

Outcome	| Share

Fully Paid |	~80%

Charged Off |	~20%

This is very typical for consumer credit portfolios.

Most loans are repaid successfully, while a smaller proportion defaults.

This imbalance will influence model evaluation metrics later, which is why accuracy alone will not be sufficient.

### Cleaning the Loan Term Variable

The `term` column represents the loan duration but is stored as text values such as:

"36 months"  
"60 months"

For modelling purposes we convert this column into a numeric variable representing the number of months.

In [17]:
df["term"] = df["term"].str.extract("(\d+)").astype(int)

df["term"].value_counts()

term
36    1020743
60     324567
Name: count, dtype: int64

### Cleaning the Interest Rate Variable

The `int_rate` column is stored as a percentage string such as:

"13.56%"

We convert this variable into a numeric value so that it can be used in statistical models.

In [18]:
pd.options.display.float_format = '{:.2f}'.format

df["int_rate"] = df["int_rate"].astype(str).str.replace("%", "").astype(float)

df["int_rate"].describe()

count   1345310.00
mean         13.24
std           4.77
min           5.31
25%           9.75
50%          12.74
75%          15.99
max          30.99
Name: int_rate, dtype: float64

### Interest Rate Distribution

The average interest rate across loans is approximately **13%**, with most loans falling between **9.7% and 16%**.

Higher interest rates typically correspond to borrowers with **lower credit grades**, reflecting risk-based pricing used by lending platforms.

### Cleaning Employment Length

The `emp_length` variable describes how long the borrower has been employed.  
However it is stored as text values such as:

"< 1 year"  
"3 years"  
"10+ years"

These values are converted into numeric years of employment so that they can be used as model features.

In [19]:
df["emp_length"] = df["emp_length"].str.replace("+ years", "", regex=False)
df["emp_length"] = df["emp_length"].str.replace("< 1 year", "0")
df["emp_length"] = df["emp_length"].str.replace(" years", "", regex=False)
df["emp_length"] = df["emp_length"].str.replace(" year", "", regex=False)

df["emp_length"] = pd.to_numeric(df["emp_length"], errors="coerce")

df["emp_length"].value_counts().sort_index()

emp_length
0.00     108061
1.00      88494
2.00     121743
3.00     107597
4.00      80556
5.00      84154
6.00      62733
7.00      59624
8.00      60701
9.00      50937
10.00    442199
Name: count, dtype: int64

From the distribution we observe:

- A large number of borrowers have **10 years of employment**, which is the highest category in the dataset.
- Many borrowers also fall in the **1–3 year employment range**, indicating relatively shorter employment histories.

Employment length is an important borrower stability indicator.  
In credit risk modelling, longer employment history is generally associated with **greater income stability and potentially lower default risk**.

### Converting Issue Date

The `issue_d` variable records the month when the loan was issued.

This variable will later be used to perform **out-of-time model validation**, where earlier loans are used for training and later loans are used for testing.

We convert this column into a proper datetime format.

In [20]:
df["issue_d"] = pd.to_datetime(df["issue_d"])

df["issue_d"]

C:/Users/ALOKTP~1/AppData/Local/Temp/xpython_56608/2368105257.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.


0         2015-12-01
1         2015-12-01
2         2015-12-01
4         2015-12-01
5         2015-12-01
             ...    
2260688   2016-10-01
2260690   2016-10-01
2260691   2016-10-01
2260692   2016-10-01
2260697   2016-10-01
Name: issue_d, Length: 1345310, dtype: datetime64[ns]

##### Quick Sanity Check of dataset

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1345310 entries, 0 to 2260697
Data columns (total 21 columns):
 #   Column               Non-Null Count    Dtype         
---  ------               --------------    -----         
 0   loan_amnt            1345310 non-null  float64       
 1   term                 1345310 non-null  int32         
 2   int_rate             1345310 non-null  float64       
 3   installment          1345310 non-null  float64       
 4   grade                1345310 non-null  object        
 5   sub_grade            1345310 non-null  object        
 6   emp_length           1266799 non-null  float64       
 7   home_ownership       1345310 non-null  object        
 8   annual_inc           1345310 non-null  float64       
 9   verification_status  1345310 non-null  object        
 10  issue_d              1345310 non-null  datetime64[ns]
 11  purpose              1345310 non-null  object        
 12  dti                  1344936 non-null  float64       
 13  fi

### Handling Missing Values

Some variables contain missing values that must be handled before modelling.

From the dataset inspection we observe missing values in a few borrower attributes such as:

- employment length
- debt-to-income ratio
- revolving credit utilisation

These variables are important predictors of borrower creditworthiness, so instead of dropping the rows we replace missing values using simple statistical imputation.

For numeric variables we use the **median**, which is more robust to extreme values than the mean.

CHECK MISSING VALUES

In [22]:
df.isnull().sum()

loan_amnt                  0
term                       0
int_rate                   0
installment                0
grade                      0
sub_grade                  0
emp_length             78511
home_ownership             0
annual_inc                 0
verification_status        0
issue_d                    0
purpose                    0
dti                      374
fico_range_low             0
fico_range_high            0
open_acc                   0
revol_bal                  0
revol_util               857
total_acc                  0
recoveries                 0
default                    0
dtype: int64

FILL MISSING VALUES

In [23]:
df["emp_length"].fillna(df["emp_length"].median(), inplace=True)
df["dti"].fillna(df["dti"].median(), inplace=True)
df["revol_util"].fillna(df["revol_util"].median(), inplace=True)

C:/Users/ALOKTP~1/AppData/Local/Temp/xpython_56608/4138265759.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


C:/Users/ALOKTP~1/AppData/Local/Temp/xpython_56608/4138265759.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=T

In [24]:
df.isnull().sum()

loan_amnt              0
term                   0
int_rate               0
installment            0
grade                  0
sub_grade              0
emp_length             0
home_ownership         0
annual_inc             0
verification_status    0
issue_d                0
purpose                0
dti                    0
fico_range_low         0
fico_range_high        0
open_acc               0
revol_bal              0
revol_util             0
total_acc              0
recoveries             0
default                0
dtype: int64

SAVE CLEAN DATASET AS PARQUET FILE FOR LOADING INTO OTHER NOTEBOOKS AFTER THIS

In [25]:
df.to_parquet("clean_lendingclub.parquet")